### `dataclass` 사용 이유

#### 1. 데이터의 구조화 (Bundling)

* **의미**: 연관된 여러 변수(이름, 나이 등)를 하나의 의미 있는 단위(`Context`)로 묶어 관리합니다.
* **효과**: 함수 인자가 줄어들고, 코드가 산만해지지 않습니다.

#### 2. 값 기반의 논리 비교 (Value Equality)

* **일반 객체**: 데이터가 같아도 메모리 주소가 다르면 `False`로 판정합니다.
* **데이터 클래스**: 내부의 실제 데이터(값)가 같으면 동일한 것으로 간주하도록 설정하여 로직의 오류를 방지합니다.

#### 3. 개발 생산성 향상 (IDE Support)

* **자동 완성**: `ctx.` 입력 시 속성 이름이 자동으로 표시되어 오타를 방지합니다.
* **타입 힌트**: 각 필드가 어떤 타입(`str`, `int`)인지 명시되어 코드 자체가 설명서 역할을 합니다.

#### 4. 코드의 안정성 (Maintainability)

* **유효성 검사**: 데이터가 생성될 때 값이 올바른지 검증하는 로직을 넣기 유리합니다.
* **불변성 제어**: 설정에 따라 데이터를 수정할 수 없게(Read-only) 만들어 예상치 못한 데이터 변경 사고를 막을 수 있습니다.

#### 5. 가독성과 디버깅 (Representation)

* **가독성**: 딕셔너리의 `ctx["user_name"]`보다 `ctx.user_name` 형태가 훨씬 읽기 쉽습니다.
* **출력 최적화**: 객체를 출력했을 때 내부 데이터를 한눈에 볼 수 있는 형식(`Context(user_name='김일남', age=20)`)을 제공하여 디버깅이 편해집니다.

---

### `dataclass`, `TypedDict`, `Pydantic` 중 어떤 것을 써야 할까요?

#### 1. 한눈에 보는 비교표

| 특징 | `dataclass` | `TypedDict` | `Pydantic` (**추천**) |
| :--- | :--- | :--- | :--- |
| **근본 형태** | 일반 객체 (Class) | 딕셔너리 (dict) | 검증 객체 (BaseModel) |
| **런타임 검증** | ❌ (타입 힌트만 제공) | ❌ (정적 분석만 가능) | ✅ (엄격한 데이터 검증) |
| **타입 강제/변환** | ❌ (지정한 타입과 달라도 들어감) | ❌ (일반 dict와 동일) | ✅ (자동 타입 변환 지원) |
| **사용처** | 내부 로직, 간단한 상태 관리 | JSON 통신, API 응답/요청 | **LLM 입출력, API 서버, 설정 정보** |
| **표준 여부** | Python 표준 라이브러리 | Python 표준 라이브러리 | 외부 라이브러리 (설치 필요) |


#### 2. 상세 설명

##### 1) dataclass (표준 라이브러리)
- **목적**: 데이터를 담는 클래스를 편하게 만들기 위함입니다.
- **특징**: `__init__`, `__repr__`, `__eq__` 등을 자동으로 생성해주어 코드가 깔끔해집니다.
- **한계**: `age: int`라고 선언해도 문자열 `"20"`을 넣으면 그대로 들어갑니다. **즉, 데이터의 "모양"만 정의할 뿐 "값의 무결성"은 보장하지 않습니다.**

##### 2) TypedDict (표준 라이브러리)
- **목적**: 딕셔너리(`{}`)의 키(Key)와 값(Value)의 타입을 명시하기 위함입니다.
- **특징**: IDE(VSCode 등)에서 자동 완성을 지원하고 오타를 줄여주지만, 실행 시점(Runtime)에는 일반 딕셔너리와 똑같이 동작합니다. 
- **한계**: 객체 지향적인 메서드 활용이 불가능하고, 데이터 검증 기능이 없습니다.

##### 3) Pydantic (외부 라이브러리 - `BaseModel`)
- **목적**: **데이터 파싱 및 엄격한 유효성 검사**입니다.
- **특징**:
    - **강력한 검증**: `age: int`에 `"20"`을 넣으면 정수 `20`으로 자동 변환(Coercion)해주고, `"abc"`를 넣으면 즉시 에러를 발생시킵니다.
    - **LLM과 찰떡궁합**: LangChain이나 OpenAI API에서 **Structured Output(구조화된 출력)**을 받을 때 표준으로 사용됩니다.
    - **JSON 변환**: `.model_dump_json()` 등을 통해 JSON으로 변환하거나 파싱하는 기능이 매우 강력합니다.


#### 3. 어떤 것을 써야 할까요?

*   **LangChain이나 AI 프로젝트를 하신다면?** 👉 무조건 **Pydantic**을 사용하세요. LLM이 내뱉는 결과를 특정 구조로 강제하거나 검증할 때 필수입니다.
*   **간단한 스크립트에서 클래스 정의가 귀찮을 때?** 👉 **dataclass**가 좋습니다. 별도 설치 없이 바로 쓸 수 있고 가볍습니다.
*   **기존에 dict 형태를 그대로 써야 하는데 타입 힌트만 주고 싶을 때?** 👉 **TypedDict**를 사용하세요.

In [7]:
class Context:
    def __init__(self, user_name: str, age: int = 99):
        self.user_name = user_name
        self.age = age

    def __eq__(self, other):
        """
        객체의 데이터 값이 동일한지 비교하는 로직을 수동으로 구현합니다.
        """
        if not isinstance(other, Context):
            return NotImplemented
        return (self.user_name == other.user_name and 
                self.age == other.age)

    def __repr__(self):
        """
        객체를 출력했을 때의 형식을 정의합니다. (선택 사항이나 dataclass는 이를 자동 생성함)
        """
        return f"Context(user_name='{self.user_name}', age={self.age})"

# 실행부
ctx1 = Context("김일남")
ctx2 = Context("김일남")

print(ctx1 == ctx2)  # 결과: True

True


In [8]:
ctx1

Context(user_name='김일남', age=99)

In [9]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    age: int = 99

ctx1 = Context("김일남")
ctx2 = Context("김일남")

print(ctx1 == ctx2)

True


In [10]:
ctx1

Context(user_name='김일남', age=99)